# FraudGraph — Serving sample export

Produces `test_sample_full.csv`: a set of held-out test transactions with their
**exact materialized 437-feature vectors** (pulled straight from `graph.pt`, so
they're identical to what the model was trained/evaluated on) plus the core
payload fields. The local serving demo replays these to show the model scoring
with full information — simulating a production feature store — while the
`/score` API keeps its minimal-field contract.

Inputs required: IEEE-CIS competition + the `fraudgraph-stage2` dataset (graph.pt).

## 0. Setup

In [4]:
!pip install -q torch_geometric

In [5]:
import os, glob, numpy as np, pandas as pd, torch

OUT = "/kaggle/working"
DATA = "/kaggle/input/competitions/ieee-fraud-detection"
GRAPH_PATH = glob.glob("/kaggle/input/**/graph.pt", recursive=True)[0]
print("graph:", GRAPH_PATH)

graph: /kaggle/input/datasets/svkayy/fraudgraph-stage2/graph.pt


## 1. Load graph + realign raw rows

The graph's transaction nodes are NOT in a simple time sort — Stage 2 sorted by
time, then re-sorted by `[card1, dt]` for the velocity computation, then sorted
back by time. Because single-column sorts aren't stable, transactions sharing a
`TransactionDT` end in a specific tie-order. We replicate that exact sequence so
row *i* here corresponds to node *i* in the graph.

In [6]:
data = torch.load(GRAPH_PATH, weights_only=False)
X = data["transaction"].x.numpy()
y = data["transaction"].y.numpy()
test_mask = data["transaction"].test_mask.numpy()
print("X:", X.shape, "| test nodes:", int(test_mask.sum()))

txn = pd.read_csv(f"{DATA}/train_transaction.csv",
                  usecols=["TransactionID", "TransactionDT", "TransactionAmt",
                           "ProductCD", "card1", "addr1", "isFraud"])
idn = pd.read_csv(f"{DATA}/train_identity.csv", usecols=["TransactionID", "DeviceInfo"])
df = txn.merge(idn, on="TransactionID", how="left")

# --- replicate Stage 2 v2 node ordering exactly ---
df = df.sort_values("TransactionDT").reset_index(drop=True)
df["card1"] = df["card1"].fillna(-999)
df["dt"] = pd.to_datetime(df["TransactionDT"], unit="s")
df = df.sort_values(["card1", "dt"])
df = df.sort_values("TransactionDT").reset_index(drop=True)
# ---------------------------------------------------

assert len(df) == X.shape[0], "row/node count mismatch"
match = float((df["isFraud"].values == y).mean())
print(f"label alignment: {match:.4%}")
assert match > 0.999, f"alignment too low ({match:.4%}) — sort sequence still differs"
print("alignment verified")

X: (590540, 437) | test nodes: 118108
label alignment: 100.0000%
alignment verified


## 2. Sample test transactions (fraud-enriched for a visible demo)

Representative sampling would give only ~3.5% fraud; for a demo that shows
APPROVE/FLAG/DECLINE spread we include more fraud than nature would.

In [7]:
rng = np.random.RandomState(42)
test_idx = np.where(test_mask)[0]
fraud_idx = test_idx[y[test_idx] == 1]
legit_idx = test_idx[y[test_idx] == 0]

n_fraud = min(200, len(fraud_idx))
n_legit = 500 - n_fraud
sel = np.concatenate([
    rng.choice(fraud_idx, n_fraud, replace=False),
    rng.choice(legit_idx, n_legit, replace=False),
])
rng.shuffle(sel)
print(f"selected {len(sel)} rows: {n_fraud} fraud + {n_legit} legit")

selected 500 rows: 200 fraud + 300 legit


## 3. Assemble + export

In [8]:
core = df.iloc[sel][["TransactionID", "isFraud", "TransactionAmt", "ProductCD",
                     "card1", "addr1", "DeviceInfo", "TransactionDT"]].reset_index(drop=True)
feat = pd.DataFrame(X[sel], columns=[f"f{i}" for i in range(X.shape[1])])
out = pd.concat([core, feat], axis=1)
out.to_csv(f"{OUT}/test_sample_full.csv", index=False)

sz = os.path.getsize(f"{OUT}/test_sample_full.csv") / 1024**2
print(f"wrote test_sample_full.csv  ({out.shape[0]} rows x {out.shape[1]} cols, {sz:.1f} MB)")
print("Download it -> local data/test_sample_full.csv")

wrote test_sample_full.csv  (500 rows x 445 cols, 2.4 MB)
Download it -> local data/test_sample_full.csv
